# 🐻 FNAF Lore — AWS Bedrock Knowledge Base Retrieval

This notebook retrieves information from an AWS Bedrock Knowledge Base built on the **Five Nights at Freddy's** lore archive, and answers questions with Claude.

## Instructions:
1. Run the **Setup** cell to install dependencies
2. Configure your Knowledge Base in the **Configuration** cell
3. Run the **Function** cell to load the retrieval function
4. Use the **Retrieval** cell to query the Knowledge Base
5. Use the **Display** cell to see formatted results
6. Use the **RAG Pipeline** cell to get Claude-generated answers

---
## 📦 Setup

In [1]:
#pip install boto3

---
## ⚙️ Configuration

**Edit this cell** to configure your Knowledge Bases and settings.

Add your Knowledge Bases to `KNOWLEDGE_BASES` dictionary with format:
```python
"KB_ID": "Friendly Name"
```

In [ ]:
# ============================================================
# ⚙️ CONFIGURATION - Edit this section with your settings
# ============================================================

# AWS Region where your Knowledge Bases are located
AWS_REGION = "us-east-1"  # e.g., "us-east-1", "ap-southeast-1"

# Knowledge Bases configuration
# Format: "KB_ID": "Friendly Name"
# Add as many as you need - duplicates will be automatically removed
KNOWLEDGE_BASES = {
    "KAXMB1SGJC": "FNAF_KB",
    # "YOUR_KB_ID_2": "Your KB Name 2",
    # "YOUR_KB_ID_3": "Your KB Name 3",
}

# Number of results to retrieve per Knowledge Base
NUMBER_OF_RESULTS = 5

# Query to search for
QUERY_TEXT = "Who is the night guard in the original Five Nights at Freddy's?"

# Output file for saving results
OUTPUT_FILE = "kb_retrievals.json"

# AWS Profile (leave as None to use default credentials)
AWS_PROFILE = None  # e.g., "my-profile" or None

# ============================================================
# Auto-generated list of KB IDs (no duplicates)
# ============================================================
KNOWLEDGE_BASE_LIST = list(KNOWLEDGE_BASES.keys())

print("✅ Configuration loaded!")
print(f"   Region: {AWS_REGION}")
print(f"   Knowledge Bases: {len(KNOWLEDGE_BASES)}")
for kb_id, kb_name in KNOWLEDGE_BASES.items():
    print(f"      • {kb_id}: {kb_name}")
print(f"   Results per KB: {NUMBER_OF_RESULTS}")

---
## 🔧 Functions

Run this cell once to load the retrieval and display functions.

In [3]:
# ============================================================
# 🔧 IMPORTS & FUNCTIONS (Run once)
# ============================================================
import boto3
import json
import textwrap

def retrieve_from_knowledge_bases(kb_list, query, num_results, region, profile_name=None):
    """
    Retrieve results from multiple Knowledge Bases.

    Args:
        kb_list: List of Knowledge Base IDs
        query: Search query text
        num_results: Number of results per KB
        region: AWS region
        profile_name: Optional AWS profile name

    Returns:
        dict: Results keyed by KB ID
    """
    # Create session with profile if specified
    if profile_name:
        session = boto3.Session(profile_name=profile_name)
        client = session.client("bedrock-agent-runtime", region_name=region)
    else:
        client = boto3.client("bedrock-agent-runtime", region_name=region)

    results = {}

    for kb in kb_list:
        try:
            resp = client.retrieve(
                knowledgeBaseId=kb,
                retrievalQuery={"text": query},
                retrievalConfiguration={
                    "vectorSearchConfiguration": {"numberOfResults": num_results}
                },
            )

            hits = resp.get("retrievalResults", [])[:num_results]
            parsed = []

            for h in hits:
                loc = h.get("location", {}) or {}
                ltype = loc.get("type")

                if ltype == "S3":
                    source = loc.get("s3Location", {}).get("uri")
                elif ltype == "WEB":
                    source = loc.get("webLocation", {}).get("url")
                else:
                    source = loc

                parsed.append({
                    "text": h.get("content", {}).get("text", ""),
                    "source": source,
                    "score": h.get("score"),
                })

            results[kb] = {
                "query": query,
                "region": region,
                "count": len(parsed),
                "results": parsed,
            }

            print(f"✅ {kb}: {len(parsed)} results")

        except Exception as e:
            results[kb] = {"error": str(e)}
            print(f"❌ {kb}: error -> {e}")

    return results


def display_results(data, knowledge_base_map, query_text, show_all_chunks=False, max_width=180):
    """
    Display formatted retrieval results.

    Args:
        data: Retrieved data dictionary
        knowledge_base_map: Dict mapping KB IDs to names
        query_text: The query that was used
        show_all_chunks: If True, show all chunks; if False, show only first
        max_width: Maximum line width for text wrapping
    """
    print("=" * 60)
    print(f"🔎 QUERY: {query_text}")
    print("=" * 60)

    for kb_id, kb_name in knowledge_base_map.items():
        if kb_id in data:
            kb_data = data[kb_id]

            if "error" in kb_data:
                print(f"\n❌ {kb_id} ({kb_name}): {kb_data['error']}")
                continue

            if kb_data.get("results"):
                print(f"\n📘 Knowledge Base: {kb_name}")
                print(f"   ID: {kb_id} | Results: {kb_data['count']}")
                print("-" * 50)

                chunks = kb_data["results"] if show_all_chunks else kb_data["results"][:1]

                for i, chunk in enumerate(chunks):
                    wrapped_text = textwrap.fill(chunk["text"], width=max_width)
                    if show_all_chunks:
                        print(f"\n[Chunk {i+1}] Score: {chunk.get('score', 'N/A')}")
                    print(wrapped_text)
                    if chunk.get("source"):
                        print(f"📎 Source: {chunk['source']}")
            else:
                print(f"\n⚠️ {kb_id} ({kb_name}): No results found")
        else:
            print(f"\n❓ {kb_id} ({kb_name}): Not in retrieved data")

print("✅ Functions loaded!")

✅ Functions loaded!


---
## 🔍 Retrieval

Run this cell to retrieve results from your Knowledge Bases.

In [4]:
# ============================================================
# 🔍 RETRIEVE FROM KNOWLEDGE BASES
# ============================================================

# Perform retrieval
retrieved_data = retrieve_from_knowledge_bases(
    KNOWLEDGE_BASE_LIST,
    QUERY_TEXT,
    NUMBER_OF_RESULTS,
    AWS_REGION,
    AWS_PROFILE,
)

# Save results to file
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(retrieved_data, f, ensure_ascii=False, indent=2)

print(f"\n💾 Results saved to: {OUTPUT_FILE}")

❌ UNKRIPGFQH: error -> An error occurred (ResourceNotFoundException) when calling the Retrieve operation: Knowledge Base with id UNKRIPGFQH does not exist

💾 Results saved to: kb_retrievals.json


---
## 📊 Display Results

View the retrieved results in a formatted way.

In [5]:
# ============================================================
# 📊 DISPLAY RESULTS
# ============================================================

# Display first chunk from each KB
display_results(
    retrieved_data,
    KNOWLEDGE_BASES,
    QUERY_TEXT,
    show_all_chunks=False  # Set to True to see all chunks
)

🔎 QUERY: ปัจจัยที่จำเป็นในการดำรงชีวิตของสัตว์น้ำ

❌ UNKRIPGFQH (FNAF_KB): An error occurred (ResourceNotFoundException) when calling the Retrieve operation: Knowledge Base with id UNKRIPGFQH does not exist


---
## 🔁 Quick Query (Optional)

Use this cell for quick queries without modifying the main configuration.

In [ ]:
# ============================================================
# 🔁 QUICK QUERY - Edit and run for new queries
# ============================================================

# Quick query settings (overrides main config for this cell only)
quick_query = "What is Remnant and how does it relate to the Afton family?"
quick_num_results = 3

# Perform quick query
quick_results = retrieve_from_knowledge_bases(
    KNOWLEDGE_BASE_LIST,
    quick_query,
    quick_num_results,
    AWS_REGION,
    AWS_PROFILE,
)

# Display results
print("\n")
display_results(
    quick_results,
    KNOWLEDGE_BASES,
    quick_query,
    show_all_chunks=True
)

---
## 🤖 RAG Pipeline with Claude (Optional)

Complete RAG pipeline: Retrieve from KB + Generate answer with Claude.

In [ ]:
# ============================================================
# 🤖 RAG PIPELINE WITH CLAUDE
# ============================================================

# Latest-generation Claude on Bedrock. Claude 4.x models are served via
# cross-region inference profiles, so the model ID is prefixed with "us.".
# Swap to Opus for the most capable (and pricier) reasoning.
DEFAULT_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
# Alternative: "us.anthropic.claude-opus-4-1-20250805-v1:0"

def rag_pipeline(query, knowledge_base_id, num_results=5, model_id=DEFAULT_MODEL_ID):
    """
    Complete RAG pipeline: Retrieve + Generate

    Args:
        query: User's question
        knowledge_base_id: AWS Bedrock Knowledge Base ID
        num_results: Number of chunks to retrieve
        model_id: Claude model to use for generation

    Returns:
        dict: Contains query, retrieved chunks, and generated answer
    """
    # Initialize clients
    bedrock_agent = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
    bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)

    print("=" * 80)
    print(f"🔍 Query: {query}")
    print("=" * 80)

    # Step 1: Retrieve from Knowledge Base
    retrieve_resp = bedrock_agent.retrieve(
        knowledgeBaseId=knowledge_base_id,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": num_results}
        },
    )

    results = retrieve_resp.get("retrievalResults", [])
    contexts = [r.get("content", {}).get("text", "") for r in results]

    print(f"\n✅ Retrieved {len(results)} chunks")

    # Step 2: Build prompt with context
    context_text = "\n\n".join([f"[Document {i+1}]\n{txt}" for i, txt in enumerate(contexts)])

    prompt = f"""You are a lore expert on the Five Nights at Freddy's (FNAF) franchise. Answer questions based ONLY on the provided context from the FNAF knowledge base.

Reference Information:
{context_text}

Question: {query}

Instructions:
- Base your answer strictly on the provided context
- Be clear, accurate, and detailed about characters, games, and lore
- Cite the relevant game or document when helpful
- If the context doesn't contain enough information to answer, state this clearly instead of guessing
- Stay in-universe and use proper FNAF terminology (animatronics, Remnant, Afton, etc.)

Answer:"""

    # Step 3: Generate answer with Claude
    request = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2000,
        "temperature": 0.5,
        "messages": [{"role": "user", "content": prompt}]
    }

    response = bedrock.invoke_model(
        modelId=model_id,
        body=json.dumps(request)
    )

    answer = json.loads(response['body'].read())['content'][0]['text']

    print("\n" + "=" * 80)
    print("🤖 ANSWER FROM CLAUDE:")
    print("=" * 80)
    print(answer)
    print("=" * 80 + "\n")

    return {
        "query": query,
        "retrieved_chunks": len(results),
        "answer": answer
    }

# Example usage:
# result = rag_pipeline(
#     query="Who is the Puppet and what is its role in the FNAF story?",
#     knowledge_base_id=KNOWLEDGE_BASE_LIST[0]  # Uses first KB from config
# )

In [ ]:
# ============================================================
# 🚀 RUN RAG PIPELINE
# ============================================================

# Uncomment and edit to run the RAG pipeline
# result = rag_pipeline(
#     query="Who is the Puppet and what is its role in the FNAF story?",
#     knowledge_base_id=KNOWLEDGE_BASE_LIST[0],
#     model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0"
# )